Add a feature that explains model predictions to users by integrating natural language generation techniques to provide human-readable justifications, dynamically adjusting explanations based on prediction confidence.

In [1]:
import pandas as pd
import numpy as np
import joblib
import math

In [2]:
df = pd.read_csv("heart_disease_train.csv")

model = joblib.load("saved_models/heart_disease_model_v1.joblib")

print("Dataset loaded:", df.shape)
print("Model loaded successfully!")

Dataset loaded: (79, 19)
Model loaded successfully!


In [3]:
features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

classes = list(model.classes_)

if len(features) != model.n_features_in_:
    raise ValueError("Model feature count mismatch!")

if not hasattr(model, "predict_proba"):
    raise ValueError("Model must support predict_proba().")

if 1 not in classes:
    raise ValueError("Positive class 1 was not found.")

print("Features:", features)
print("Classes:", classes)

Features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Classes: [np.int64(0), np.int64(1)]


In [4]:
background = df[features].dropna().sample(
    n=min(10, len(df)),
    random_state=42
).reset_index(drop=True)

print("Background records:", len(background))

Background records: 10


In [5]:
def calculate_shap(user_input):

    n = len(features)
    x = np.array([user_input[f] for f in features])

    total_masks = 2 ** n
    background_array = background.to_numpy()

    coalition_data = np.tile(
        background_array,
        (total_masks, 1)
    )

    for mask in range(total_masks):
        for j in range(n):
            if mask & (1 << j):
                start = mask * len(background)
                end = start + len(background)
                coalition_data[start:end, j] = x[j]

    coalition_df = pd.DataFrame(
        coalition_data,
        columns=features
    )

    positive_index = classes.index(1)

    probabilities = model.predict_proba(
        coalition_df
    )[:, positive_index]

    values = probabilities.reshape(
        total_masks, len(background)
    ).mean(axis=1)

    shap_values = []

    for j in range(n):

        contribution = 0

        for mask in range(total_masks):

            if mask & (1 << j):
                continue

            size = bin(mask).count("1")

            weight = (
                math.factorial(size) *
                math.factorial(n - size - 1)
                / math.factorial(n)
            )

            new_mask = mask | (1 << j)

            contribution += weight * (
                values[new_mask] - values[mask]
            )

        shap_values.append(contribution)

    return np.array(shap_values), values[0]

In [6]:
def generate_explanation(user_input):

    input_df = pd.DataFrame([user_input], columns=features)

    prediction = model.predict(input_df)[0]

    probabilities = model.predict_proba(input_df)[0]

    confidence = float(max(probabilities) * 100)

    positive_probability = float(
        probabilities[classes.index(1)] * 100
    )

    shap_values, baseline = calculate_shap(user_input)

    if confidence >= 80:
        confidence_level = "High"
        confidence_message = (
            "The model assigns a relatively high probability "
            "to its predicted class."
        )

    elif confidence >= 60:
        confidence_level = "Moderate"
        confidence_message = (
            "The model shows moderate confidence. "
            "The prediction should be interpreted cautiously."
        )

    else:
        confidence_level = "Low"
        confidence_message = (
            "The model has relatively low confidence. "
            "Its prediction is uncertain."
        )

    direction = 1 if prediction == 1 else -1

    contributions = []

    for feature, value in zip(features, shap_values):

        support = value * direction

        contributions.append({
            "Feature": feature,
            "Input Value": user_input[feature],
            "Contribution": round(float(value), 4),
            "Effect": (
                "Supports prediction"
                if support > 0
                else "Opposes prediction"
                if support < 0
                else "Neutral"
            ),
            "Strength": abs(float(value))
        })

    contribution_df = pd.DataFrame(contributions)

    contribution_df = contribution_df.sort_values(
        "Strength",
        ascending=False
    )

    result = (
        "Positive" if prediction == 1 else "Negative"
    )

    explanation = f"""
MODEL PREDICTION EXPLANATION

Predicted Result: {result}

Prediction Confidence: {confidence:.2f}%

Confidence Level: {confidence_level}

Positive-Class Probability: {positive_probability:.2f}%

{confidence_message}

FEATURE CONTRIBUTIONS:
"""

    for _, row in contribution_df.head(3).iterrows():

        explanation += (
            f"\n- {row['Feature']} = {row['Input Value']}: "
            f"{row['Effect']} "
            f"(Contribution: {row['Contribution']:+.4f})"
        )

    explanation += """

INTERPRETATION:
These feature contributions explain how the model's
input values influence its prediction relative to
the selected background dataset.

IMPORTANT:
This is a machine learning explanation, not a medical
diagnosis or proof of medical causation.
"""

    return explanation, contribution_df

In [7]:
user_input = {}

for feature in features:
    user_input[feature] = float(
        input(f"Enter {feature}: ")
    )

print("\nUser inputs received successfully.")


User inputs received successfully.


In [8]:
explanation, contribution_df = generate_explanation(user_input)

print(explanation)


MODEL PREDICTION EXPLANATION

Predicted Result: Negative

Prediction Confidence: 79.18%

Confidence Level: Moderate

Positive-Class Probability: 20.82%

The model shows moderate confidence. The prediction should be interpreted cautiously.

FEATURE CONTRIBUTIONS:

- trestbps = 45.0: Supports prediction (Contribution: -0.5606)
- oldpeak = 34.0: Opposes prediction (Contribution: +0.2760)
- thalach = 645.0: Supports prediction (Contribution: -0.1608)

INTERPRETATION:
These feature contributions explain how the model's
input values influence its prediction relative to
the selected background dataset.

IMPORTANT:
This is a machine learning explanation, not a medical
diagnosis or proof of medical causation.



c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [9]:
display(
    contribution_df[
        ["Feature", "Input Value", "Contribution", "Effect"]
    ]
)

,Feature,Input Value,Contribution,Effect
1,trestbps,45.0,-0.5606,Supports prediction
4,oldpeak,34.0,0.2760,Opposes prediction
3,thalach,645.0,-0.1608,Supports prediction
2,chol,464.0,0.0837,Opposes prediction
0,age,45.0,-0.0344,Supports prediction
